In [ ]:
import yaml
import einops
from einops import rearrange
import torch

## Clay encoding module

Create custom metadata that conveys the following:
- HH and HV SAR bands
- The pixel spacing is 40 metres
- The wavelength is C-band (~5.4 cm, same as all Sentinel-1)
- The normalisation mean for HH is X, std is Y (from dataset_stats.json)
- The normalisation mean for HV is X, std is Y (from dataset_stats.json)

### Explore metadata

In [ ]:
with open("../../configs/metadata.yaml") as f:
    metadata = yaml.safe_load(f)

# Find an existing SAR entry to use as reference
print(yaml.dump(metadata.get("sentinel-1-rtc"), default_flow_style=False))

band_order:
- vv
- vh
bands:
  mean:
    vh: -18.673
    vv: -12.113
  std:
    vh: 8.017
    vv: 8.314
  wavelength:
    vh: 4.0
    vv: 3.5
gsd: 10



In [ ]:
# Create a new entry for sentinel-1-ew, using the same structure as sentinel-1-rtc
session = boto3.Session(profile_name="spk_data")
s3 = session.client("s3")
bucket = "prescient-ice-data"
prefix = "training_data/ai4arctic/statistics/"

stats = json.loads(s3.get_object(
    Bucket=bucket,
    Key=prefix + "dataset_stats.json"
)['Body'].read())

sentinel1_ew_entry = {
    "sentinel-1-ew": {
        "band_names": ["hh", "hv"],
        "gsd": 40,
        "wavelengths": [5.405, 5.405], # nominal C-band SAR placeholders, consistent with Clay's built-in SAR entries
        "mean": [
            stats["nersc_sar_primary"]["mean"],
            stats["nersc_sar_secondary"]["mean"],
        ],
        "std": [
            stats["nersc_sar_primary"]["std"],
            stats["nersc_sar_secondary"]["std"],
        ],
    }
}

print(yaml.dump(sentinel1_ew_entry, default_flow_style=False))

sentinel-1-ew:
  band_names:
  - hh
  - hv
  gsd: 40
  mean:
  - -14.50393
  - -24.699312
  std:
  - 5.662948
  - 4.750692
  wavelengths:
  - 5.405
  - 5.405



In [ ]:
# Load existing metadata
with open("../../configs/metadata.yaml") as f:
    metadata = yaml.safe_load(f)

# Check it doesn't already exist
if "sentinel-1-ew" in metadata:
    print("Entry already exists")
else:
    metadata["sentinel-1-ew"] = sentinel1_ew_entry["sentinel-1-ew"]
    
    with open("../../configs/metadata.yaml", "w") as f:
        yaml.dump(metadata, f, default_flow_style=False)
    
    print("Entry added and saved")

Entry already exists


In [ ]:
with open("../../configs/metadata.yaml") as f:
    metadata = yaml.safe_load(f)

print(yaml.dump(metadata["sentinel-1-ew"], default_flow_style=False))

band_names:
- hh
- hv
gsd: 40
mean:
- -14.50393
- -24.699312
std:
- 5.662948
- 4.750692
wavelengths:
- 5.405
- 5.405



In [ ]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
CHECKPOINT_PATH = "../../clay-v1.5.ckpt"
METADATA_PATH = "../../configs/metadata.yaml"
module = ClayMAEModule.load_from_checkpoint(
    checkpoint_path=CHECKPOINT_PATH,
    model_size="large",
    mask_ratio=0.0,
    shuffle=False,
    metadata_path=METADATA_PATH
)
module.eval()
module = module.to(DEVICE)
print(f"Clay loaded on {DEVICE}")
print(f"Embedding dim: {module.model.encoder}")

Clay loaded on cpu
Embedding dim: Encoder(
  (patch_embedding): DynamicEmbedding(
    (weight_generator): WavesTransformer(
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0): TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
            )
            (linear1): Linear(in_features=128, out_features=2048, bias=True)
            (dropout): Dropout(p=0, inplace=False)
            (linear2): Linear(in_features=2048, out_features=128, bias=True)
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
            (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
            (dropout1): Dropout(p=0, inplace=False)
            (dropout2): Dropout(p=0, inplace=False)
          )
        )
      )
      (fc_weight): Linear(in_features=128, out_features=65536, bias=True)
      (fc_bias): Line

In [ ]:
sar_meta = metadata["sentinel-1-ew"]

chip = next(yield_chips())
patch_tokens, cls_token = encode_chip(chip)

print(f"Class token shape:    {cls_token.shape}")     # (1, 1024)
print(f"Patch tokens shape:   {patch_tokens.shape}")  # (1, 1024, 32, 32)

In [ ]:
cls_embedding = encoded[:, 0, :].cpu().numpy()  # (1, 1024)

patch_embedding = einops.rearrange(
    encoded[:, 1:, :].detach().cpu().numpy(),
    "b (h w) d -> b d h w",
    h=32, w=32,
)  # (1, 1024, 32, 32)

print(f"CLS embedding shape:   {cls_embedding.shape}")
print(f"Patch embedding shape: {patch_embedding.shape}")

CLS embedding shape:   (1, 1024)
Patch embedding shape: (1, 1024, 32, 32)


In [ ]:
def encode_chip(chip):
    """Encode one chip with Clay.

    Builds the datacube from a chip dict produced by yield_chips(), runs the
    encoder with mask_ratio=0.0, and returns the patch and class tokens as
    numpy arrays.

    Returns
    -------
    patch_tokens : np.ndarray  (1, 1024, 32, 32)
    cls_token    : np.ndarray  (1, 1024)
    """
    sar_mean = torch.tensor(sar_meta["mean"], dtype=torch.float32)
    sar_std  = torch.tensor(sar_meta["std"],  dtype=torch.float32)
    sar_norm = (torch.tensor(chip["sar"], dtype=torch.float32)
                - sar_mean[:, None, None]) / sar_std[:, None, None]

    datacube = {
        "pixels": sar_norm.unsqueeze(0).to(DEVICE),
        "waves":  torch.tensor(sar_meta["wavelengths"], dtype=torch.float32).to(DEVICE),  # (C,) 1D
        "gsd":    torch.tensor(float(sar_meta["gsd"]),  dtype=torch.float32).to(DEVICE),  # scalar
        "time":   torch.tensor(chip["time_encoding"],   dtype=torch.float32).unsqueeze(0).to(DEVICE),
        "latlon": torch.tensor(chip["latlon_encoding"], dtype=torch.float32).unsqueeze(0).to(DEVICE),
    }

    with torch.no_grad():
        encoded, *_ = module.model.encoder(datacube)

    cls_token    = encoded[:, 0, :].cpu().numpy()                                        # (1, 1024)
    patch_tokens = einops.rearrange(
        encoded[:, 1:, :].cpu().numpy(),
        "b (h w) d -> b d h w", h=32, w=32,
    )                                                                                     # (1, 1024, 32, 32)

    return patch_tokens, cls_token